In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# Load cleaned data (for IDs and later merging)
df = pd.read_csv("../data/processed/cleaned_data.csv")

# Load scaled features (for model input)
X_scaled = np.load("../data/processed/X_scaled.npy")

# Load feature names (optional, for interpretation)
with open("../data/processed/feature_names.txt") as f:
    feature_names = [line.strip() for line in f]

print("Data shape:", X_scaled.shape)
print("Number of features:", len(feature_names))


In [1]:
from sklearn.ensemble import IsolationForest

# Select scaled features for anomaly detection
scaled_features = [f"{col}_scaled" for col in numeric_features]
X_scaled_for_if = data_scaled[scaled_features]

# Build and fit the Isolation Forest model
iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # assume ~5% anomalies
    random_state=42
)

iso_labels = iso_model.fit_predict(X_scaled_for_if)

# Add anomaly labels back into dataset
data_scaled["isolation_forest_anomaly"] = iso_labels


NameError: name 'numeric_features' is not defined

In [ ]:
anomalies = data_scaled[data_scaled["isolation_forest_anomaly"] == -1]
normal = data_scaled[data_scaled["isolation_forest_anomaly"] == 1]

print("Anomaly count:", len(anomalies))
print("Normal count:", len(normal))


In [ ]:
feature_means = pd.DataFrame({
    "Normal Mean": normal[numeric_features].mean(),
    "Anomaly Mean": anomalies[numeric_features].mean()
})

feature_means


In [ ]:
top_anomaly_senders = anomalies["sender"].value_counts().head(5)
top_anomaly_recipients = anomalies["recipient"].value_counts().head(5)

top_anomaly_senders, top_anomaly_recipients


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.scatter(normal["amount0"], normal["amount1"], s=10,c="pink" ,label="Normal", alpha=0.5)
plt.scatter(anomalies["amount0"], anomalies["amount1"], s=20,c="skyblue" ,label="Anomaly", alpha=0.9)
plt.xlabel("amount0")
plt.ylabel("amount1")
plt.legend()
plt.title("Isolation Forest Anomalies in Swap Space")
plt.show()


In [ ]:
anomalies_per_time = anomalies.groupby("timeStamp_dt").size()

plt.figure(figsize=(6,3))
anomalies_per_time.plot(kind="line")
plt.title("Anomalies Over Time")
plt.ylabel("Count")
plt.xlabel("Time")
plt.show()


In [ ]:
# Isolation Forest anomaly scores (higher = more anomalous after sign flip)
anomaly_scores = iso_model.decision_function(X_scaled_for_if)

data_scaled["isolation_score"] = -anomaly_scores  # flip so larger = more anomalous

print("Isolation Forest anomaly score summary:\n")
print(data_scaled["isolation_score"].describe())


In [ ]:
print("\nTop 5 highest-risk anomalies:\n")
cols_to_show = ["sender", "recipient", "amount0", "amount1", "liquidity", "gasPrice", "isolation_score"]
print(data_scaled.sort_values("isolation_score", ascending=False).head(5)[cols_to_show])

In [ ]:
from sklearn.cluster import KMeans

# Use scaled features for anomaly clustering
anom_scaled = anomalies[[f"{col}_scaled" for col in numeric_features]]

kmeans = KMeans(n_clusters=2, random_state=42)
anomalies.loc[:, "anomaly_group"] = kmeans.fit_predict(anom_scaled)


# Optional: see how many in each group
anomalies["anomaly_group"].value_counts()


In [ ]:
cluster_profiles = anomalies.groupby("anomaly_group")[numeric_features].mean()
cluster_profiles


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in numeric_features:
    plt.figure(figsize=(6,3),facecolor="pink")
    sns.boxplot(data=anomalies, x="anomaly_group", y=col,color="skyblue")
    plt.title(f"{col}: cluster behaviour difference")
    plt.show()


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled_for_if)

data_scaled["pc1"] = pca_data[:, 0]
data_scaled["pc2"] = pca_data[:, 1]

plt.figure(figsize=(6, 4))
plt.scatter(
    data_scaled[data_scaled["isolation_forest_anomaly"] == 1]["pc1"],
    data_scaled[data_scaled["isolation_forest_anomaly"] == 1]["pc2"],
    s=20, c="hotpink" , alpha=0.4, label="Normal"
)
plt.scatter(
    data_scaled[data_scaled["isolation_forest_anomaly"] == -1]["pc1"],
    data_scaled[data_scaled["isolation_forest_anomaly"] == -1]["pc2"],
    s=20,c="skyblue" ,alpha=0.8, label="Anomaly"
)
plt.legend()
plt.title("PCA Visualization of Isolation Forest Anomalies")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()
